# POS classes distance


## Get embadding distance

In [ ]:
import javalang
import re
import numpy as np

In [ ]:
def parse_java_file_ast(path):    
    with open(path, encoding='utf-8') as javafile:
        lines = javafile.read()
        tree = javalang.parse.parse(lines)
    return tree
def getPackage(tree):
    return tree.package.name

def getImports(tree):
    return [node.path for node in tree.imports]

def getMethods(tree):
    return [method.name for method in tree.types[0].methods]

def getFields(tree):
    return [node.declarators[0].name for node in tree.types[0].fields]
def getName(tree):
    return tree.types[0].name

In [ ]:
#pos classes
data=[]
app=[]
entity=[]
utility=[]
inp = open("data/posapplication.txt", "r")
applicationF = inp.read().splitlines()
inp.close()
for c in applicationF:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data.append(elem)
    app.append(elem)
inp = open("data/posutility.txt", "r")
utilityF = inp.read().splitlines()
inp.close()
for c in utilityF:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data.append(elem)
    utility.append(elem)
inp = open("data/posentity.txt", "r")
entityF = inp.read().splitlines()
inp.close()
for c in entityF:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data.append(elem)
    entity.append(elem)



In [ ]:
entity=['employeemodel', 'invoicemodel', 'saledao', 'purchasemodel', 'productmodel', 'productdao', 'categorymodel', 'invoicedao', 'product', 'purchase', 'sale', 'employeedao', 'employee', 'salesmodel', 'supplier', 'categorydao', 'suppliermodel', 'purchasedao', 'item', 'supplierdao', 'invoice', 'payment']
app=['saleinterface', 'employeeinterface', 'suppliercontroller', 'supplieraddcontroller', 'reportviewcontroller', 'employeecontroller', 'categoryaddcontroller', 'mainapp', 'confirmcontroller', 'productaddcontroller', 'categoryinterface', 'category', 'printinvoice', 'categorycontroller', 'productinterface', 'poscontroller', 'suppliereditcontroller', 'logincontroller', 'employeeaddcontroller', 'purchaseinterface', 'invoicecontroller', 'reportcontroller', 'purchaseaddcontroller', 'categoryeditcontroller', 'purchasecontroller', 'supplierinterface', 'employeeeditcontroller', 'productcontroller', 'salescontroller', 'reportinterface', 'admincontroller', 'producteditcontroller']
utility=['hibernateutil']
typedClasses=entity+app+utility

### Word2vec

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
def getLemm(string):
    return lemmatizer.lemmatize(string)

In [ ]:
from gensim.models import Word2Vec 
import gensim.downloader as api

v2w_model = api.load('word2vec-google-news-300')

In [ ]:
def embadW2V(string):
    
    return v2w_model[string]

In [ ]:
from scipy.spatial import distance
from scipy import spatial

def distanceSementic(arr1, arr2):
    return spatial.distance.cosine(arr1, arr2)
    #return distance.euclidean(arr1, arr2)

In [ ]:
def readFile(fileName):
        fileObj = open(fileName, "r") #opens the file in read mode
        lines = fileObj.read().splitlines() #puts the file into an array
        fileObj.close()
        return lines

In [ ]:
stopW=['handle', 'cancel',"title",'parent', "cell","bean" ,'loader','stage', 'pressed', 'dragged','view', 'box', 'initialize',"total","view","image","icon","offset","node","scene",'duration', 'drawer', 'nav','load', 'data', 'is', 'empty', 'all',"static","cascade",'transaction', 'override',"join","one","description","strategy","generation","override",'persistence', 'generated',"io","projection","property","commit","dao","this",'style', 'menu',"begin","column","translate","on","selected",'name',"png","logo", 'string', 'name',"table","exception","contains","filter","controller","implement","button","session","hibernate","array","org","save","clear","boolean","init","remove","entity","observable","double","length","alert","action","field","bundle","show","root","list","index","text","return","wait","lower","true","false","java","util","long","collection","interface","layout","value","valid","is""value","type","model","public","private","id","error","void","not","int","float","for","set","catch","try","javafx","import","class","com","package","if","else",'null',"no","delete","add","edit","get","new","open","close","mouse","event","window","throw"]

#stopW=[]

In [ ]:
rx = re.compile(r'(?<=[a-z])(?=[A-Z])')
allWords=dict()
for elem in typedClasses:
    filelines=readFile('./newdir/'+elem.lower()+'.java')
    strings=[]
    for s in filelines:
        sentence=" ".join(re.findall("[a-zA-Z]+", s))
        words=sentence.split()
        for w in words:
            if(w not in stopW):
                strings.append(w)
    nstrings = [re.findall('.[^A-Z]*', string) for string in strings]
    all=[]
    for ns in nstrings:
        all=all+ns
    #lemm + to lower cases
    treatedW=[]
    embads=[]
    w=[]
    for a in all:
        treatedW.append(getLemm(a.lower()))
    for t in treatedW:
        if(t in v2w_model and t not in stopW and len(t)>1):
            if t in allWords:
                allWords[t]=allWords[t]+1
            else:
                allWords[t]=1
print(len(allWords))
reccurency= 0
for a in allWords:
    reccurency+=allWords[a]
print(reccurency)
for a in allWords:
    allWords[a]=allWords[a]/reccurency


In [ ]:
rx = re.compile(r'(?<=[a-z])(?=[A-Z])')
embadDict=dict()
wListClass=dict()
aSIF=0.001
for elem in typedClasses:
    tree=parse_java_file_ast('./newdir/'+elem.lower()+'.java')
    #methods=getMethods(tree)
    #fields=getFields(tree)
    #strings=methods+fields
    #strings.append(getName(tree))
    filelines=readFile('./newdir/'+elem.lower()+'.java')
    strings=[]
    for s in filelines:
        sentence=" ".join(re.findall("[a-zA-Z]+", s))
        words=sentence.split()
        for w in words:
            if(w not in stopW):
                strings.append(w)
    nstrings = [re.findall('.[^A-Z]*', string) for string in strings]
    all=[]
    for ns in nstrings:
        all=all+ns
    #lemm + to lower cases
    treatedW=[]
    embads=[]
    w=[]
    for a in all:
        treatedW.append(getLemm(a.lower()))
    for t in treatedW:
        if(t in v2w_model and t not in stopW and len(t)>1):
            w.append(t)
            coefSIF=aSIF/(aSIF+allWords[t])
            emzz=embadW2V(t)
            #embads.append(coefSIF*emzz)
            embads.append(emzz)
    wListClass[elem]=w
    npmatrix=np.matrix(embads)
    
    if(len(npmatrix) > 1): 
        meanmatrix=npmatrix.mean(0)
        
        embadDict[elem]=meanmatrix
    else:
        embadDict[elem]=npmatrix


## Create dict with relation distance 

### Calculate static distance

In [ ]:

import csv
distances=[]
staticDistanceMin=99999
staticDistanceMax=0
with open('./data/POS_GRAPH.csv', newline='') as posgraph:
    reader = csv.reader(posgraph, delimiter=';')
    for row in reader:
        rowClass=[]
        class1=row[0].lower().split('.')[-1]
        class2= row[1].lower().split('.')[-1]
        if(class1 in embadDict and class2 in embadDict ):
            rowClass.append(class1)
            rowClass.append(class2)
            rowClass.append(row[2])
            if(float(row[2])>staticDistanceMax):
                staticDistanceMax=float(row[2])
            if(float(row[2])<staticDistanceMin):
                staticDistanceMin=float(row[2])
            #data.append(1-distanceSementic(embadDict[class1],embadDict[class2]))
            distances.append(rowClass)
staticDistanceMin=0
#for d in distances:
    #d[2]=(float(d[2])-staticDistanceMin)/(staticDistanceMax-staticDistanceMin)


### Calculate semantic distance

In [ ]:
semanticDistanceMin=99999
semanticDistanceMax=0
fullConnectG=[]
for c in typedClasses:
    for tc in typedClasses:
        d=[]
        d.append(c)
        d.append(tc)
        d.append(0)
        dsemandic=1-distanceSementic(embadDict[c],embadDict[tc])
        if(dsemandic>semanticDistanceMax):
            semanticDistanceMax=dsemandic
        if(dsemandic<semanticDistanceMin):
            semanticDistanceMin=dsemandic   
        d.append(dsemandic)
        fullConnectG.append(d)
for fcg in fullConnectG:
    for d in distances:
        if(d[0]==fcg[0] and d[1]==fcg[1]):
            fcg[2]=d[2]
            break
    fcg[3]=(fcg[3]-semanticDistanceMin)/(semanticDistanceMax-semanticDistanceMin)


## Create typed services

In [ ]:
from operator import itemgetter
from networkx.algorithms.community.centrality import girvan_newman 
from networkx.algorithms.community import kernighan_lin_bisection
import itertools
import matplotlib.pyplot as plt
import networkx as nx
def heaviest(G):
    u, v, w = max(G.edges(data="weight"), key=itemgetter(2))
    return (u, v)
def weightSR(s,r):
    weightRelation=100
    weightSementic=100
    return (weightRelation*(float(r)))+(weightSementic*s)

In [ ]:
from cdlib import algorithms, viz
coms = algorithms.louvain(GApp, weight='weight',resolution=0.5)
print(coms.communities)
applicationServices=coms.communities


In [ ]:
GEntity = nx.Graph()
for e in entity:
    GEntity.add_node(e)
for d in distances:
    if(d[0] in entity and d[1] in entity):
        GEntity.add_edge(d[0], d[1], weight=float(d[2]))

coms = algorithms.louvain(GEntity, weight='weight',resolution=0.5)
print(coms.communities)
entityServices=coms.communities

In [ ]:
GUtility = nx.Graph()
for u in utility:
    GUtility.add_node(u)
for d in distances:
    if(d[0] in utility and d[1] in utility):
        GUtility.add_edge(d[0], d[1], weight=float(d[2]))

coms = algorithms.louvain(GUtility, weight='weight',resolution=0.5)
print(coms.communities)
utilityServices=coms.communities

### fine tune clusters

In [ ]:
import numpy as np

def fineTuneCluster(service,indexS, services):
    scoreService=[]
    for s in services:
        scoreService.append(0)
    for d in distances:
        if(d[0]==service and d[1]!=service):
            for i,s in enumerate(services):
                for ones in s:
                    if(d[1]==ones ):
                        scoreService[i]+=d[2]
                        break
        elif(d[1]==service and d[0]!=service ):
            for i,s in enumerate(services):
                for ones in s:
                    if(d[0]==ones ):
                        scoreService[i]+=d[2]
                        break
    maxscore=max(scoreService)
    if(maxscore>0):
        maxIndex=scoreService.index(max(scoreService))  
        indices=[]
        for i, x in enumerate(scoreService):
            if x == maxscore:
                indices.append(i)
        if(len(indices)==1):
            services[maxIndex].append(service)
            services.remove([service])
    applicationServices
    return services
#entity
arrayEntityServices=[]
for es in entityServices:
    arrayEntityServices.append(es)
entityServices = arrayEntityServices
for i,es in enumerate(entityServices):
    if(len(es)<2): 
        entityServices=fineTuneCluster(es[0],i,entityServices)
print("***",entityServices)

#application
arrayApplicationServices=[]
for es in applicationServices:
    arrayApplicationServices.append(es)
applicationServices = arrayApplicationServices
for i,es in enumerate(applicationServices):
    if(len(es)<2): 
        applicationServices=fineTuneCluster(es[0],i,applicationServices)
print("***",applicationServices)

#utility
arrayUtilityServices=[]
for es in utilityServices:
    arrayUtilityServices.append(es)
utilityServices = arrayUtilityServices
#for i,es in enumerate(utilityServices):
    #if(len(es)<2): 
        #utilityServices=fineTuneCluster(es[0],i,utilityServices)
print("***",utilityServices)

## Creating graph from services

In [ ]:
def embadServices(arrC):
    total=0
    embadS=0
    for c in arrC: 
        embadC=embadDict[c]*len(wListClass[c])
        total+=len(wListClass[c])
        embadS=embadS+embadC
       
    return embadS/total

In [ ]:
services=dict()
servicesEmbad=dict()
classService=dict()
for appS in applicationServices:
    name="Application"+appS[0]
    services[name]=appS
    servicesEmbad[name]=embadServices(appS)
    for a in appS:
        classService[a]=name
for appS in entityServices:
    name="Entity"+appS[0]
    services[name]=appS
    servicesEmbad[name]=embadServices(appS)
    for a in appS:
        classService[a]=name
for appS in utilityServices:
    name="utility"+appS[0]
    services[name]=appS
    servicesEmbad[name]=embadServices(appS)
    for a in appS:
        classService[a]=name

In [ ]:
distanceServices=[]
for d in distances:
    class1=d[0]
    class2=d[1]
    dis=float(d[2])
    if(class1 in classService and class2 in classService):
        S1=classService[class1] 
        S2=classService[class2] 
        find=False
        for ds in distanceServices:
            if((ds[0]==S1 and ds[1]==S2)or(ds[1]==S1 and ds[0]==S2)):
                ds[2]=ds[2]+dis
                ds[3]=ds[3]+distanceSementic(servicesEmbad[S1],servicesEmbad[S2])
                find=True
        if(not find):
            sementic=distanceSementic(servicesEmbad[S1],servicesEmbad[S2])
            distanceServices.append([S1,S2,dis,sementic])


## Microservice clustering

In [ ]:
from networkx.algorithms.community import k_clique_communities


In [ ]:
GServices = nx.Graph()
def poidsSR(s,r):
    weightRelation=0.1
    weightSementic=100
    if(not(s==0)):
        s=1/s
    return (weightRelation*float(r))+(weightSementic*s)
f = open("weight.txt", "a")
for d in distanceServices: 
    allWeight=poidsSR(d[3],d[2])
    GServices.add_edge(d[0], d[1], weight=allWeight)
    d.append(allWeight)
    f.write(str(d[0])+" "+ str(d[1])+" "+str(allWeight)+"\n")

f.close()


In [ ]:
nodeList=list(GServices.nodes)
print(len(nodeList))
adjMatrix=[]
INF=999
for node1 in nodeList:
    array=[]
    for node2 in nodeList:
        array.append(INF)
    adjMatrix.append(array)
for i,node in enumerate(nodeList):
    adjs=GServices.adj[node]
    for adjOne in adjs:
        index=nodeList.index(adjOne)
        for d in distanceServices: 
            if(d[0]==adjOne and d[1]==node):
                adjMatrix[i][index]=100/d[2]
                break
            if(d[1]==adjOne and d[0]==node):
                adjMatrix[i][index]=100/d[2]
                break



In [ ]:
# Floyd Warshall Algorithm in python


# The number of vertices
nV = len(GServices.nodes)
INF = 999
print(nV)
# Algorithm implementation
def floyd_warshall(G):
    distance = list(map(lambda i: list(map(lambda j: j, i)), G))
    # Adding vertices individually
    for k in range(nV):
        for i in range(nV):
            for j in range(nV):
                
                distance[i][j] = min(distance[i][j], distance[i][k] + distance[k][j])
    #print_solution(distance)
    return distance


# Printing the solution
def print_solution(distance):
    for i in range(nV):
        print(nodeList[i])
    for i in range(nV):
        print("----",nodeList[i])
        for j in range(nV):
            
            if(distance[i][j] == 999):
                print("INF", end=" ")
            else:
                print(distance[i][j], end="  ")
        print(" ")


G = adjMatrix
distances=floyd_warshall(G)
print_solution(distances)

In [ ]:
for i,ni in enumerate(nodeList):
    for j,nj in enumerate(nodeList):
        ds=distanceSementic(servicesEmbad[ni],servicesEmbad[nj])
        ds=ds*2
        distances[i][j]=ds+float(distances[i][j])

In [ ]:
#C fuzzy
import math

m=3
coef=2/(m-1)
allwij=dict()
def calcWeight(i,c):
    wij=0
    dij=distances[i][c]
    for iApp in indexApplication:            
        v=dij/distances[i][indexApplication[iApp]]
        val=math.pow(v, coef)
        wij+=val
    return wij
#get applciation services indexes 
indexApplication=dict()
for i,a in enumerate(nodeList):
    if a.startswith("Application"):
        indexApplication[a]=i   
# calculate weights of all 
print(nodeList)

In [ ]:
resultedMS=dict()
for index,iApp in enumerate(indexApplication):
    wij=[]
    resMs=[]
    print("** ",index)
    for s in services[iApp]:
        print("$ ",s)
        resMs.append(s)

    for i,a in enumerate(nodeList): 
        if(a!=iApp):
            c=calcWeight(i,indexApplication[iApp])
            val=1/c
            wij.append(val)
            #print(a,iApp,val*100)
            if(val*100>9):
                for s in services[a]:
                    print(s)
                    resMs.append(s)
    resultedMS[index]=resMs
resultedMS.pop(10)